# Ray Train Distributed Patch Prediction Demo

This notebook demonstrates distributed prediction using Ray Train, where each worker processes a mutually exclusive subset of database shards, generates synthetic patch predictions, and synchronizes with a barrier after each round.

In [11]:
import ray
from ray import train
from ray.train import get_context
from ray.train.collective import barrier
import numpy as np
import psycopg
from psycopg.rows import dict_row
import os
import random
import datetime
from db_client import CitusHeadClient

# Configuration
NUM_WORKERS = 4  # Set as needed
PATCHES_PER_BATCH = 50  # Number of patches per batch

# DB connection (reuse constants from db_client)
from constants import CITUS_HEAD_HOST, CITUS_HEAD_PORT, CITUS_HEAD_DB, CITUS_HEAD_USER, CITUS_HEAD_PASSWORD

def get_db_conn():
    return psycopg.connect(
        host=CITUS_HEAD_HOST,
        port=CITUS_HEAD_PORT,
        dbname=CITUS_HEAD_DB,
        user=CITUS_HEAD_USER,
        password=CITUS_HEAD_PASSWORD,
        autocommit=True,
        row_factory=dict_row
    )

In [12]:
def get_patch_shards():
    """Return a sorted list of all non-empty patch table shard IDs."""
    with get_db_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT shardid FROM pg_dist_shard WHERE logicalrelid = 'patch'::regclass ORDER BY shardid;")
            return [row['shardid'] for row in cur.fetchall()]


class ShardDataset:
    """Loads real patches from assigned Citus shard tables and yields them in batches."""
    def __init__(self, assigned_shards: list, patches_per_batch: int):
        self.assigned_shards = assigned_shards
        self.patches_per_batch = patches_per_batch

    def __iter__(self):
        client = CitusHeadClient()
        patches = client.fetch_patches_by_shards(self.assigned_shards)
        for i in range(0, len(patches), self.patches_per_batch):
            yield patches[i:i + self.patches_per_batch]

In [13]:
def insert_predictions(batch: list):
    """Bulk-insert predictions for a batch of patches into pred_patch_latest. Append-only."""
    values = [
        (
            patch['patch_id'],
            random.uniform(0.0, 1.0),   # embed_x (synthetic)
            random.uniform(0.0, 1.0),   # embed_y (synthetic)
            random.randint(0, 10),      # grid_cell_i (synthetic)
            random.randint(0, 10),      # grid_cell_j (synthetic)
            datetime.datetime.now(),
            int(patch['label_class_id']),
        )
        for patch in batch
    ]
    with get_db_conn() as conn:
        with conn.cursor() as cur:
            cur.executemany("""
                INSERT INTO pred_patch_latest
                    (patch_id, embed_x, embed_y, grid_cell_i, grid_cell_j, event_ts, label_class_id)
                VALUES (%s, %s, %s, %s, %s, %s, %s);
            """, values)

In [14]:
def rotate_pred_tables():
    """
    Called by rank 0 between barriers at the end of each cycle.
    Drops pred_patch_last, promotes pred_patch_latest -> pred_patch_last,
    and creates a fresh empty pred_patch_latest.
    """
    with get_db_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("DROP TABLE IF EXISTS pred_patch_last CASCADE;")
            cur.execute("ALTER TABLE pred_patch_latest RENAME TO pred_patch_last;")
            cur.execute("""
                CREATE TABLE pred_patch_latest (
                    patch_id BIGINT PRIMARY KEY,
                    embed_x FLOAT NOT NULL,
                    embed_y FLOAT NOT NULL,
                    grid_cell_i SMALLINT NOT NULL,
                    grid_cell_j SMALLINT NOT NULL,
                    event_ts TIMESTAMP NOT NULL,
                    label_class_id SMALLINT NOT NULL REFERENCES label_class(label_class_id)
                );
            """)
            cur.execute("SELECT create_distributed_table('pred_patch_latest', 'patch_id');")
    print("[Rank 0] Table rotation complete: pred_patch_latest is fresh, pred_patch_last holds previous cycle.")


def train_worker(config):
    context = get_context()
    rank = context.get_world_rank()
    world_size = context.get_world_size()
    print(f"[Worker {rank}] Starting. World size: {world_size}")

    all_shards = config['all_shards']
    assigned_shards = [s for i, s in enumerate(all_shards) if i % world_size == rank]
    print(f"[Worker {rank}] Assigned {len(assigned_shards)} shards: {assigned_shards}")

    cycle = 0
    while True:
        cycle += 1
        print(f"[Worker {rank}] Starting cycle {cycle}.")

        dataset = ShardDataset(assigned_shards, config['patches_per_batch'])

        num_batches = 0
        for batch_num, batch in enumerate(dataset):
            print(f"[Worker {rank}] Cycle {cycle} — batch {batch_num + 1} ({len(batch)} patches).")
            insert_predictions(batch)
            num_batches += 1
        print(f"[Worker {rank}] Cycle {cycle} done ({num_batches} batches). Waiting at barrier.")

        # Barrier 1: all workers finished inserting for this cycle
        barrier()

        # Rank 0 rotates tables while other workers wait
        if rank == 0:
            rotate_pred_tables()

        # Barrier 2: rotation complete, all workers resume
        barrier()
        print(f"[Worker {rank}] Cycle {cycle} rotation complete. Starting next cycle.")

In [15]:
# Main Ray Train execution cell
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig

if not ray.is_initialized():
    ray.init()

# Fetch all patch shard IDs for distribution across workers
all_shards = get_patch_shards()
print(f"Total patch shards: {len(all_shards)}")

trainer = TorchTrainer(
    train_loop_per_worker=train_worker,
    train_loop_config={
        "all_shards": all_shards,
        "patches_per_batch": PATCHES_PER_BATCH,
    },
    scaling_config=ScalingConfig(
        num_workers=NUM_WORKERS,
        use_gpu=False,
    ),
)

result = trainer.fit()
print("Ray Train Results:")
print(result)

Total patch shards: 32


(TrainController pid=186062) Requesting resources: {'CPU': 1} * 4
(TrainController pid=186062) Attempting to start training worker group of size 4 with the following resources: [{'CPU': 1}] * 4
(PlacementGroupCleaner pid=186160) Exception in thread PlacementGroupCleanerMonitor:
(PlacementGroupCleaner pid=186160) Traceback (most recent call last):
(PlacementGroupCleaner pid=186160)   File "/home/jackson/.local/share/uv/python/cpython-3.11.15-linux-x86_64-gnu/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
(PlacementGroupCleaner pid=186160)     self.run()
(PlacementGroupCleaner pid=186160)   File "/home/jackson/.local/share/uv/python/cpython-3.11.15-linux-x86_64-gnu/lib/python3.11/threading.py", line 982, in run
(PlacementGroupCleaner pid=186160)     self._target(*self._args, **self._kwargs)
(PlacementGroupCleaner pid=186160)   File "/home/jackson/PatchSorter/prototyping/ray_dl_loop_prototype/.venv/lib/python3.11/site-packages/ray/util/tracing/tracing_helper.py", line 461, i

WorkerGroupError: Training failed due to worker errors:
[Rank 0,1,2,3 Error Snippet]:
Traceback (most recent call last):
  File "/tmp/ipykernel_172560/2447935345.py", line 46, in train_worker
  File "/tmp/ipykernel_172560/3314724883.py", line 17, in insert_predictions
  File "/home/jackson/PatchSorter/prototyping/ray_dl_loop_prototype/.venv/lib/python3.11/site-packages/psycopg/cursor.py", line 148, in executemany
    raise ex.with_traceback(None)
psycopg.errors.UniqueViolation: duplicate key value violates unique constraint "pred_patch_latest_pkey_105952"
DETAIL:  Key (patch_id)=(8) already exists.
